# 01. Calling an Azure OpenAI Model via LangChain

This notebook demonstrates the simplest possible way to talk to a model deployed in **Azure AI Foundry** from **LangChain**: point LangChain's `ChatOpenAI` wrapper at the Foundry project's OpenAI-compatible `/openai/v1` endpoint, supply a deployment name as the "model", and call `.invoke()`. It belongs to Section 04 of this course, "Agent frameworks on top of Azure AI Foundry" — the section that layers third-party orchestration frameworks (LangChain, LangGraph) on top of the Azure AI Foundry model layer covered in earlier sections.

**Difficulty: Beginner**

**Important framing:** LangChain is a vendor-neutral, third-party Python framework. It is **not** part of the AI-102 (Designing and Implementing a Microsoft Azure AI Solution) exam syllabus. What *is* exam-relevant here is the underlying Azure service being called — an Azure AI Foundry model deployment — and the authentication pattern used to reach it. Treat everything LangChain-specific as "how to build agents productively," and everything about the endpoint/deployment/auth as "what AI-102 actually tests."

## Prerequisites

**pip3 packages** (add to your environment; not all are in the repo root `requirements.txt` yet):
```bash
pip3 install langchain-openai python-dotenv
```
`langchain-core` is a transitive dependency of `langchain-openai` and is already in the root `requirements.txt`, along with `langchain`, `langchain-community`, `langchain-classic`, and `langchain-text-splitters`.

**Azure resources required:**
- An Azure AI Foundry project with at least one chat-completion model deployed (the original script uses a deployment named `gpt-5.4`; use whatever deployment name exists in your own project).

**Environment variables** (add these to the root `.env` alongside the existing `AZURE_AI_PROJECT_ENDPOINT` / `AZURE_AI_MODEL_DEPLOYMENT` pair used throughout `11_azure_ai_foundry/`):
```
AZURE_AI_OPENAI_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/openai/v1
AZURE_AI_OPENAI_API_KEY=<your-foundry-api-key>
AZURE_AI_MODEL_DEPLOYMENT=<your-chat-deployment-name>
```
Note this endpoint is deliberately different in shape from `AZURE_AI_PROJECT_ENDPOINT` (which points at `/api/projects/<project-name>` for the Agents/Projects SDKs). Here we're hitting the plain OpenAI-compatible `/openai/v1` surface that Foundry exposes for any OpenAI-SDK-shaped client, LangChain included. Also note this script authenticates with an **API key**, not `DefaultAzureCredential`/Entra ID like the `11_azure_ai_foundry/` labs — Foundry model endpoints support either, and LangChain's `ChatOpenAI` only knows how to send a bearer key, not an Entra token.

## What You'll Learn

- How to point LangChain's `ChatOpenAI` at a non-OpenAI, OpenAI-API-compatible endpoint using `base_url`
- Why an Azure AI Foundry deployment name is passed as the `model` parameter
- The minimal shape of a LangChain chat call: construct a client once, call `.invoke()` with a prompt, read `.content`
- Where LangChain's abstraction ends and the underlying Azure REST API begins

### Step 1 — Imports and configuration

`ChatOpenAI` from `langchain_openai` is LangChain's wrapper around any endpoint that speaks the OpenAI Chat Completions / Responses wire format. Azure AI Foundry deployments expose exactly that shape at `<resource>/openai/v1`, so no Azure-specific LangChain package is needed for this simple case.

We load configuration from environment variables via `python-dotenv` instead of hardcoding the endpoint, deployment name, and API key as the original script did — this keeps secrets out of source control and out of the notebook itself.

💡 **Framework note:** LangChain doesn't know or care that this endpoint belongs to Azure AI Foundry — it just sees "an OpenAI-compatible base URL." The Azure-specific pieces are the URL shape (`.services.ai.azure.com/openai/v1`) and the deployment name you put in the `model` field. On the AI-102 exam, this same endpoint is what you'd call directly with the `openai` Python SDK or `azure-ai-projects`' `get_openai_client()` — LangChain is just one more client on top of it.

🔄 **Alternatives:** For exam-scoped work, prefer the raw `openai` SDK pointed at the same `base_url`, or `AIProjectClient(...).get_openai_client()` from `azure-ai-projects` (as used throughout `11_azure_ai_foundry/`) — both avoid pulling in LangChain as a dependency when you don't need its agent/chain abstractions.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

endpoint = os.getenv("AZURE_AI_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT")
api_key = os.getenv("AZURE_AI_OPENAI_API_KEY")


### Step 2 — Build the client

`ChatOpenAI` is constructed once with `base_url`, `api_key`, and `model`. Nothing happens over the network yet — this just configures the client for later calls.

💡 **Framework note:** the `model` parameter here is doing double duty. In vanilla OpenAI it names a model family (`gpt-4o`); against an Azure AI Foundry `/openai/v1` endpoint it must instead be the **deployment name** you gave the model when you deployed it in the Foundry portal or via `azure-ai-projects`. Passing a model *family* name here that doesn't match an actual deployment will fail.

🔄 **Alternatives:** you can also configure `ChatOpenAI` via the `AZURE_OPENAI_*` environment variables that `langchain_openai.AzureChatOpenAI` reads automatically — but that class targets the classic `azure.openai.com` resource shape (with `api_version`), not Foundry's newer `/openai/v1` unified surface used here, so plain `ChatOpenAI` with an explicit `base_url` is the correct choice for this endpoint style.

In [ ]:
client = ChatOpenAI(
    base_url=endpoint,
    api_key=api_key,
    model=deployment_name,
)


### Step 3 — Invoke the model

`.invoke()` sends a single user message and blocks until the full response comes back (as opposed to `.stream()`, which would yield tokens incrementally). The return value is an `AIMessage`; `.content` is the plain-text answer.

💡 **Framework note:** under the hood this is a single POST to the Foundry deployment's chat/responses endpoint — identical to what you'd get calling `openai.chat.completions.create(...)` directly against the same `base_url`.

🔄 **Alternatives:** swap `.invoke()` for `.stream()` to get incremental tokens for a more responsive UI, or `.batch()` to send several independent prompts concurrently.

In [ ]:
response = client.invoke(
    "What are the three main benefits of using managed AI endpoints in the cloud?"
)
print(response.content)


## Summary

This notebook built a `ChatOpenAI` client pointed at an Azure AI Foundry model deployment's OpenAI-compatible endpoint, then made a single blocking `.invoke()` call and printed the model's text response. The pattern is deliberately minimal — no tools, no memory, no agent loop — establishing the baseline that later notebooks in this section (tool-calling agents, RAG agents, telemetry) build on top of.

## Try It Yourself

1. **Easy:** Change the prompt to ask a different question and observe how the response changes. Try setting `temperature=0` vs `temperature=1` on the `ChatOpenAI` constructor and compare outputs across repeated runs.
2. **Intermediate:** Replace `.invoke()` with `.stream()` and print each chunk as it arrives to see token-by-token streaming.
3. **Advanced:** Rewrite this notebook using the raw `openai` Python SDK (`OpenAI(base_url=..., api_key=...)` + `.chat.completions.create(...)`) instead of LangChain, and compare how much code LangChain's wrapper actually saves you for this single-call case.